# 04_heart_rctd_three_samples

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/RCTD_fullref_UMI30 (3 samples).ipynb`

Three explicit UMI_min=30 reruns: Aged1/Aged2/Young1. Young2 effective UMI_min must be checked in saved object.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 2 (one-based); original execution count 3.

In [ ]:
library(spacexr)
library(Matrix)
library(AnnotationDbi)
library(org.Mm.eg.db)
library(arrow)
library(future)
library(Seurat)
library(BPCells)

plan(multicore, workers = 18)
options(future.globals.maxSize = 250 * 1024^3)

Source cell 3 (one-based); original execution count 4.

In [ ]:
## ===============================
## 1) load Seurat object & counts_joined
## ===============================
heart <- readRDS("heart_mergeobj_preProject_full_sketchHarmony.rds")

# counts_joined を作って取得（Seurat v5 layer）
heart <- JoinLayers(
  object = heart,
  assay  = "RNA",
  layers = "counts",
  new    = "counts_joined"
)

counts <- GetAssayData(
  object = heart,
  assay  = "RNA",
  layer  = "counts_joined"
)

# 確認点1：サンプル数
table(heart$sample)

Source cell 4 (one-based); original execution count 5.

In [ ]:
## ===============================
## 2) reference の読み込み（あなたのファイル名に合わせて変更）
## ===============================
reference_rctd <- readRDS("reference_rctd_heart.rds")

Source cell 5 (one-based); original execution count 6.

In [ ]:
## ===============================
## Heart_Aged_1  (cutoff30 を「本当に」効かせる版)
## 変更点：create.RCTD に UMI_min = 30 を明示
## ===============================

sample <- "Heart_Aged_1"
pos_path <- "/work/Visium/Heart_combined_samples/Tissue_Position/Heart_Aged_1/tissue_positions.parquet"
nCount_cutoff <- 30

# 1) cell抽出（nCount>=30）
cells <- colnames(heart)[heart$sample == sample & heart$nCount_RNA >= nCount_cutoff]
stopifnot(length(cells) > 0)
length(cells)  # ログに残す

# 2) coords
pos <- read_parquet(pos_path)
coords_raw <- data.frame(
  x = pos$pxl_col_in_fullres,
  y = pos$pxl_row_in_fullres,
  row.names = pos$barcode
)
cells_base <- sub("^.*_(s_008um_)", "\\1", cells)
idx <- match(cells_base, rownames(coords_raw))
stopifnot(all(!is.na(idx)))

coords_sample <- coords_raw[idx, , drop = FALSE]
rownames(coords_sample) <- cells
stopifnot(identical(rownames(coords_sample), cells))

# 3) counts_use（列抜き）
counts_use <- counts[, cells, drop = FALSE]

# 4) ENSMUSG -> SYMBOL（集約）
ens_keys <- unique(rownames(counts_use))
map_df <- AnnotationDbi::select(
  org.Mm.eg.db,
  keys    = ens_keys,
  keytype = "ENSEMBL",
  columns = "SYMBOL"
)
map_df <- map_df[!is.na(map_df$SYMBOL) & map_df$SYMBOL != "", c("ENSEMBL","SYMBOL")]
map_df <- map_df[!duplicated(map_df$ENSEMBL), ]
ens2sym <- setNames(map_df$SYMBOL, map_df$ENSEMBL)

ens_in <- intersect(rownames(counts_use), names(ens2sym))
stopifnot(length(ens_in) > 0)

M_ens   <- counts_use[ens_in, , drop = FALSE]
sym_ids <- ens2sym[ens_in]
symbols <- unique(sym_ids)

A <- sparseMatrix(
  i = match(sym_ids, symbols),
  j = seq_along(sym_ids),
  x = 1,
  dims = c(length(symbols), length(sym_ids))
)

M_sym <- A %*% M_ens
rownames(M_sym) <- symbols

# 5) SpatialRNA
stopifnot(identical(colnames(M_sym), rownames(coords_sample)))
nUMI_sym <- Matrix::colSums(M_sym)

spatialRNA_sample <- SpatialRNA(
  coords = coords_sample,
  counts = M_sym,
  nUMI   = nUMI_sym
)

# 6) create.RCTD（fullref保持）→ run.RCTD → save
myRCTD <- create.RCTD(
  spatialRNA = spatialRNA_sample,
  reference  = reference_rctd,
  max_cores  = 8,
  keep_reference = TRUE,
  UMI_min = 30
)

myRCTD <- run.RCTD(myRCTD, doublet_mode = "doublet")

saveRDS(myRCTD, file = paste0("RCTD_fullref_cutoff30_", sample, ".rds"))

# 7) 確認（結果が入ったか）
c(
  n_spots   = ncol(myRCTD@spatialRNA@counts),
  n_results = nrow(myRCTD@results$results_df),
  ref_cells = ncol(myRCTD@reference@counts),
  min_nUMI  = min(myRCTD@spatialRNA@nUMI)
)
table(myRCTD@results$results_df$spot_class)

## 8) メモリ掃除（次のサンプルへ）
rm(pos, coords_raw, cells_base, idx, coords_sample,
   counts_use, ens_keys, map_df, ens2sym, ens_in,
   M_ens, sym_ids, symbols, A, M_sym, nUMI_sym, spatialRNA_sample,
   myRCTD)
gc()


Source cell 6 (one-based); original execution count 7.

In [ ]:
## ===============================
## Heart_Aged_2  (cutoff30 を「本当に」効かせる版)
## 変更点：create.RCTD に UMI_min = 30 を明示
## ===============================

sample <- "Heart_Aged_2"
pos_path <- "/work/Visium/Heart_combined_samples/Tissue_Position/Heart_Aged_2/tissue_positions.parquet"
nCount_cutoff <- 30

# 1) cell抽出（nCount>=30）
cells <- colnames(heart)[heart$sample == sample & heart$nCount_RNA >= nCount_cutoff]
stopifnot(length(cells) > 0)
length(cells)  # ログに残す

# 2) coords
pos <- read_parquet(pos_path)
coords_raw <- data.frame(
  x = pos$pxl_col_in_fullres,
  y = pos$pxl_row_in_fullres,
  row.names = pos$barcode
)
cells_base <- sub("^.*_(s_008um_)", "\\1", cells)
idx <- match(cells_base, rownames(coords_raw))
stopifnot(all(!is.na(idx)))

coords_sample <- coords_raw[idx, , drop = FALSE]
rownames(coords_sample) <- cells
stopifnot(identical(rownames(coords_sample), cells))

# 3) counts_use（列抜き）
counts_use <- counts[, cells, drop = FALSE]

# 4) ENSMUSG -> SYMBOL（集約）
ens_keys <- unique(rownames(counts_use))
map_df <- AnnotationDbi::select(
  org.Mm.eg.db,
  keys    = ens_keys,
  keytype = "ENSEMBL",
  columns = "SYMBOL"
)
map_df <- map_df[!is.na(map_df$SYMBOL) & map_df$SYMBOL != "", c("ENSEMBL","SYMBOL")]
map_df <- map_df[!duplicated(map_df$ENSEMBL), ]
ens2sym <- setNames(map_df$SYMBOL, map_df$ENSEMBL)

ens_in <- intersect(rownames(counts_use), names(ens2sym))
stopifnot(length(ens_in) > 0)

M_ens   <- counts_use[ens_in, , drop = FALSE]
sym_ids <- ens2sym[ens_in]
symbols <- unique(sym_ids)

A <- sparseMatrix(
  i = match(sym_ids, symbols),
  j = seq_along(sym_ids),
  x = 1,
  dims = c(length(symbols), length(sym_ids))
)

M_sym <- A %*% M_ens
rownames(M_sym) <- symbols

# 5) SpatialRNA
stopifnot(identical(colnames(M_sym), rownames(coords_sample)))
nUMI_sym <- Matrix::colSums(M_sym)

spatialRNA_sample <- SpatialRNA(
  coords = coords_sample,
  counts = M_sym,
  nUMI   = nUMI_sym
)

# 6) create.RCTD（fullref保持）→ run.RCTD → save
myRCTD <- create.RCTD(
  spatialRNA = spatialRNA_sample,
  reference  = reference_rctd,
  max_cores  = 8,
  keep_reference = TRUE,
  UMI_min = 30
)

myRCTD <- run.RCTD(myRCTD, doublet_mode = "doublet")

saveRDS(myRCTD, file = paste0("RCTD_fullref_cutoff30_", sample, ".rds"))

# 7) 確認（結果が入ったか）
c(
  n_spots   = ncol(myRCTD@spatialRNA@counts),
  n_results = nrow(myRCTD@results$results_df),
  ref_cells = ncol(myRCTD@reference@counts),
  min_nUMI  = min(myRCTD@spatialRNA@nUMI)
)
table(myRCTD@results$results_df$spot_class)

## 8) メモリ掃除（次のサンプルへ）
rm(pos, coords_raw, cells_base, idx, coords_sample,
   counts_use, ens_keys, map_df, ens2sym, ens_in,
   M_ens, sym_ids, symbols, A, M_sym, nUMI_sym, spatialRNA_sample,
   myRCTD)
gc()


Source cell 8 (one-based); original execution count 9.

In [ ]:
## ===============================
## Heart_Young_1  (cutoff30 を「本当に」効かせる版)
## 変更点：create.RCTD に UMI_min = 30 を明示
## ===============================

sample <- "Heart_Young_1"
pos_path <- "/work/Visium/Heart_combined_samples/Tissue_Position/Heart_Young_1/tissue_positions.parquet"
nCount_cutoff <- 30

# 1) cell抽出（nCount>=30）
cells <- colnames(heart)[heart$sample == sample & heart$nCount_RNA >= nCount_cutoff]
stopifnot(length(cells) > 0)
length(cells)  # ログに残す

# 2) coords
pos <- read_parquet(pos_path)
coords_raw <- data.frame(
  x = pos$pxl_col_in_fullres,
  y = pos$pxl_row_in_fullres,
  row.names = pos$barcode
)
cells_base <- sub("^.*_(s_008um_)", "\\1", cells)
idx <- match(cells_base, rownames(coords_raw))
stopifnot(all(!is.na(idx)))

coords_sample <- coords_raw[idx, , drop = FALSE]
rownames(coords_sample) <- cells
stopifnot(identical(rownames(coords_sample), cells))

# 3) counts_use（列抜き）
counts_use <- counts[, cells, drop = FALSE]

# 4) ENSMUSG -> SYMBOL（集約）
ens_keys <- unique(rownames(counts_use))
map_df <- AnnotationDbi::select(
  org.Mm.eg.db,
  keys    = ens_keys,
  keytype = "ENSEMBL",
  columns = "SYMBOL"
)
map_df <- map_df[!is.na(map_df$SYMBOL) & map_df$SYMBOL != "", c("ENSEMBL","SYMBOL")]
map_df <- map_df[!duplicated(map_df$ENSEMBL), ]
ens2sym <- setNames(map_df$SYMBOL, map_df$ENSEMBL)

ens_in <- intersect(rownames(counts_use), names(ens2sym))
stopifnot(length(ens_in) > 0)

M_ens   <- counts_use[ens_in, , drop = FALSE]
sym_ids <- ens2sym[ens_in]
symbols <- unique(sym_ids)

A <- sparseMatrix(
  i = match(sym_ids, symbols),
  j = seq_along(sym_ids),
  x = 1,
  dims = c(length(symbols), length(sym_ids))
)

M_sym <- A %*% M_ens
rownames(M_sym) <- symbols

# 5) SpatialRNA
stopifnot(identical(colnames(M_sym), rownames(coords_sample)))
nUMI_sym <- Matrix::colSums(M_sym)

spatialRNA_sample <- SpatialRNA(
  coords = coords_sample,
  counts = M_sym,
  nUMI   = nUMI_sym
)

# 6) create.RCTD（fullref保持）→ run.RCTD → save
myRCTD <- create.RCTD(
  spatialRNA = spatialRNA_sample,
  reference  = reference_rctd,
  max_cores  = 8,
  keep_reference = TRUE,
  UMI_min = 30
)

myRCTD <- run.RCTD(myRCTD, doublet_mode = "doublet")

saveRDS(myRCTD, file = paste0("RCTD_fullref_cutoff30_", sample, ".rds"))

# 7) 確認（結果が入ったか）
c(
  n_spots   = ncol(myRCTD@spatialRNA@counts),
  n_results = nrow(myRCTD@results$results_df),
  ref_cells = ncol(myRCTD@reference@counts),
  min_nUMI  = min(myRCTD@spatialRNA@nUMI)
)
table(myRCTD@results$results_df$spot_class)

## 8) メモリ掃除（次のサンプルへ）
rm(pos, coords_raw, cells_base, idx, coords_sample,
   counts_use, ens_keys, map_df, ens2sym, ens_in,
   M_ens, sym_ids, symbols, A, M_sym, nUMI_sym, spatialRNA_sample,
   myRCTD)
gc()
